# Oxta Contábil — treino e benchmark local em AMD

Notebook oficial do piloto de produto NSOS. A arquitetura permitida é **Mamba + Attention + OxtaMem + QAT ternária de 2 bits**. MoE, KAN, TTT, CHRASS e Slender permanecem experimentais e são desligados explicitamente pelo benchmark.

Este notebook apenas orquestra scripts versionados para evitar que uma cópia de código dentro do `.ipynb` fique desatualizada. O runtime alvo é HIP/ROCm na AMD Radeon RX 7600.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

NSOS_ROOT = Path.cwd().resolve()
while NSOS_ROOT != NSOS_ROOT.parent and not (NSOS_ROOT / 'scripts' / 'train_curriculum.py').exists():
    NSOS_ROOT = NSOS_ROOT.parent
if not (NSOS_ROOT / 'scripts' / 'train_curriculum.py').exists():
    raise RuntimeError('Abra o notebook dentro do monorepo OXN/nsos.')

BUILD_DIR = NSOS_ROOT / 'build-codex-hip'
DATA_DIR = NSOS_ROOT / 'artifacts' / 'oxta_contabil_amd' / 'data'
REPORT = NSOS_ROOT / 'artifacts' / 'oxta_contabil_amd' / 'benchmark' / 'rx7600_product_architecture_ablation400.json'
print('NSOS:', NSOS_ROOT)
print('HIP build:', BUILD_DIR)
print('Dados:', DATA_DIR)

## 1. Dados públicos

- BR-TaxQA-R: domínio tributário brasileiro (CC BY 4.0).
- FineWeb-2 `por_Latn`: pré-treino em português (ODC-By 1.0); o downloader aplica sobreamostragem fiscal.
- bAbI QA1: ablação arquitetural reproduzível (CC BY 3.0), revisão e hashes pinados.

BR-TaxQA-R e FineWeb-2 são material de produto; bAbI não mede contabilidade e serve somente para isolar memória/arquitetura.

In [ ]:
download = [
    sys.executable, str(NSOS_ROOT / 'scripts' / 'oxta_contabil' / 'download_datasets.py'),
    '--output-dir', str(DATA_DIR),
    '--include', 'br_taxqa,culturax_ptbr',
    '--target-bytes', '64MB',
    '--culturax-shards', '1',
]
subprocess.run(download, cwd=NSOS_ROOT, check=True)

## 2. Ablação na GPU AMD

A execução usa `strict_gpu_execution=True`; um fallback silencioso para CPU causa erro. O benchmark compara os quatro braços com o mesmo seed, dados e orçamento. Use 400 passos para a triagem e 1.500 passos por braço apenas para a campanha confirmatória isolada.

In [ ]:
steps = 400
benchmark = [
    sys.executable, '-B',
    str(NSOS_ROOT / 'scripts' / 'oxta_contabil' / 'benchmark_product_architecture.py'),
    '--build-dir', str(BUILD_DIR), '--data-dir', str(DATA_DIR),
    '--steps', str(steps), '--batch-size', '32', '--eval-batch-size', '128',
    '--arms', 'mamba', 'attention', 'hybrid', 'hybrid_qat',
    '--hybrid-seeds', '11', '--ablation-seed', '11',
    '--report', str(REPORT),
]
env = os.environ.copy()
env['PATH'] = r'C:\TheRock\build\bin;C:\TheRock\build\lib\llvm\bin;' + env.get('PATH', '')
subprocess.run(benchmark, cwd=NSOS_ROOT, env=env, check=True)

In [ ]:
result = json.loads(REPORT.read_text(encoding='utf-8'))
rows = result['results']
for row in rows:
    print(f"{row['arm']:12s} acc={row['accuracy']:.3f}  "
          f"examples/s={row['train_examples_per_second']:.1f}  params={row['parameters']:,}")
print('\nOxtaMem:', json.dumps(result['oxtamem'], indent=2, ensure_ascii=False))

## 3. Regra de decisão

Não declarar SOTA com uma única tarefa ou semente. A arquitetura só avança para produção quando: (1) o híbrido superar os braços puros em múltiplas sementes e em dados contábeis separados por hash de conteúdo; (2) o QAT preservar qualidade aceitável; (3) OxtaMem for avaliado também com recuperação semântica não estruturada; e (4) latência, memória e reprodutibilidade passarem nos gates de produção.